# Suite RFL — Reflection

`Reflection` binds the schema data classes to reference object schemas, so that a store's objects are schemas: each
kind of schema and each property, branch or part, parameter and adjacency is an object of its own, linked by
`Schemas.Members` and `Schemas.Types`, and listed by the store's catalog. They are read through the visitor protocols,
written and read as snapshots, validated and built as any objects are.

In [ ]:
import { JSON, Modules, Proxies, Reflection as R, Schemas as S, Validators as V } from "@mbse/schemas/Framework";
import { assert, equal, map, raises, same, text } from "./support.js";

const Phone = new S.OfObject.Builder().name("Phone").properties(text("number")).create();
const Listing = new S.OfRelation.Builder().name("Listing").links("directory", "contact").properties(text("key")).unique(
  "contact").create();
const Contact = new S.OfObject.Builder().name("Contact").ref().description("A person").properties(
  text("name"), (p) => p.name("home").of(Phone).description("Where to call"),
  (p) => p.name("tags").of((t) => t.as_indexed((i) => i.of((x) => x.as_native(String)).extent({ minimum: 0n, maximum: 9n })))).relations(
  (a) => a.name("listings").of(Listing).me("contact")).create();
const SCHEMAS: unknown[] = [Contact, Phone, Listing];
const n = new S.Form.Data("variable", { name: "n" });
const Word = new S.OfNative.Builder().name("Word").type(BigInt).parameters((p) => p.name("n").of((t) => t.as_native(BigInt))).bits(n).create();
const Reach = new S.OfUnion.Builder().name("Reach").branches((b) => b.name("phone").of(Phone), (b) => b.name("other").of(Phone)).create();
const Both = new S.OfIntersection.Builder().name("Both").parts((b) => b.name("a").of(Phone), (b) => b.name("b").of(Phone)).create();
const Byte = new S.OfApply.Builder().name("Byte").of(Word).arguments(8n).create();
const Keyed = new S.OfIndexed.Builder().name("Keyed").key((t) => t.as_native(String)).of(Byte).create();
const Pair = new S.OfRelation.Builder().name("Pair").links("a", "b").create(); // no uniques
const Bare = new S.OfApply.Builder().name("Bare").of(Word).create(); // no arguments
const KINDS: unknown[] = [Word, Reach, Both, Byte, Keyed, Pair, Bare];

## RFL-01 · A store of schemas: its catalog lists them, and every kind and element is an object of its own

In [ ]:
{
  const store = R.store(SCHEMAS);
  const catalog = store.singleton("Schemas.Catalog") as unknown as R.Catalog;
  assert(same(catalog.schemas, SCHEMAS) && catalog.schema_name() === "Schemas.Catalog" && catalog.owner() === null);
  assert(same(store.extent("Schemas.Object"), [Contact, Phone]) && same(store.extent("Schemas.Relation"), [Listing])); // the schemas themselves
  assert(same(store.extent("Schemas.Property").map((p: any) => p.name), ["name", "home", "tags", "number", "key"])); // reached through members
  assert(same(store.extent("Schemas.Adjacency").map((a: any) => a.name), ["listings"]) && store.extent("Schemas.Indexed").length === 1);
  const home = Contact.properties.get("home") as S.OfProperty.Data;
  assert(Contact.schema_name() === "Schemas.Object" && home.schema_name() === "Schemas.Property" && Contact.identity() === Contact.identity()
    && Contact.identity() !== home.identity() && Contact.owner() === null);
  assert(same([...R.META.values()].map((meta) => meta.name), [
    "Schemas.Native", "Schemas.Object", "Schemas.Union", "Schemas.Intersection", "Schemas.Indexed", "Schemas.Apply",
    "Schemas.Relation", "Schemas.Property", "Schemas.Member", "Schemas.Parameter", "Schemas.Adjacency"]));
  assert([...R.META.values()].every((meta) => meta.ref && meta.validate().length === 0));
  assert(same([...store.names()].sort(), ["Schemas.Catalog", ...[...R.META.values()].map((m) => m.name as string), "Schemas.Members",
    "Schemas.Types", "Schemas.Listed"].sort()));
  assert((R.store().singleton("Schemas.Catalog") as unknown as R.Catalog).schemas.length === 0); // an empty catalog
  const kinds = R.store(KINDS);
  assert(same(kinds.extent("Schemas.Native").map((s: any) => s.name), ["Word", null, null, null])); // and three unnamed: n's, number's, a key's
  assert(same(kinds.extent("Schemas.Parameter").map((p: any) => p.name), ["n"]) && same(kinds.extent("Schemas.Member").map((m: any) => m.name), [
    "phone", "other", "a", "b"]));
}

## RFL-02 · Read through the visitor protocols: natives, values in their module form, members and types by role

In [ ]:
function members(value: any): unknown[] {
  return (V.entries_of(value).get("members") ?? []).map((e) => [e.values.get("role"), e.values.get("index"), e.targets.get("member")]);
}

function types(value: any): unknown[] {
  return (V.entries_of(value).get("types") ?? []).map((e) => [e.values.get("role"), e.targets.get("type")]);
}

{
  assert(equal(V.properties_of(Contact), map({ name: "Contact", description: "A person", ref: true })));
  assert(equal(members(Contact), [["properties", 0n, Contact.properties.get("name")], ["properties", 1n, Contact.properties.get("home")],
    ["properties", 2n, Contact.properties.get("tags")], ["adjacencies", 0n, Contact.adjacencies.get("listings")]]));
  const home = Contact.properties.get("home") as S.OfProperty.Data;
  assert(equal(V.properties_of(home), map({ name: "home", description: "Where to call" })) && equal(types(home), [["type", Phone]]));
  const tags = Contact.properties.get("tags")?.type as S.OfIndexed.Data;
  assert(equal((V.properties_of(tags).get("extent") as any).values, map({ minimum: 0n, maximum: 9n }))); // a value, in its module form
  assert(equal(types(tags), [["item", tags.item]]) && equal(V.properties_of(tags.item), map({ format: "basic", token: "str" })));
  assert(equal((V.properties_of(Listing).get("links") as any).values, ["directory", "contact"]));
  assert(equal((V.properties_of(Listing).get("uniques") as any).values.map((u: any) => u.values), [["contact"]]));
  const listings = Contact.adjacencies.get("listings") as S.OfAdjacency.Data;
  assert(equal(V.properties_of(listings), map({ name: "listings", me: "contact" })) && equal(types(listings), [["relation", Listing]]));
  assert(!V.properties_of(Word).has("bits") && (V.properties_of(Word).get("terms") as any).values.get("bits").values.get("kind") === "variable");
  assert(equal(members(Word), [["parameters", 0n, Word.parameters.get("n")]]) && equal(types(Word.parameters.get("n")), [["type", Word.parameters.get("n")?.type]]));
  assert(equal(members(Reach), [["branches", 0n, Reach.branches[0]], ["branches", 1n, Reach.branches[1]]]));
  assert(same(members(Both).map((m: any) => m[0]), ["parts", "parts"]) && equal(types(Both.parts[0]), [["type", Phone]]));
  assert(equal(types(Byte), [["of", Word]]) && (V.properties_of(Byte).get("arguments") as any).values[0].values.get("name") === "n");
  assert(equal(types(Keyed), [["item", Byte], ["key", Keyed.key]]));
  assert(!V.properties_of(Pair).has("uniques") && !V.properties_of(Bare).has("arguments") && equal(types(Bare), [["of", Word]]));
  assert(V.entries_of(Phone).get("types") === undefined && (V.entries_of(Phone).get("members") as any)[0].values.get("role") === "properties");
}

## RFL-03 · A snapshot of a store of schemas, read back: schemas whose modules are the same, shared and recursive ones too

In [ ]:
{
  const Node = new S.OfObject.Builder().name("Node").properties(text("label")).create();
  new S.OfObject.Builder(Node).properties((p) => p.name("next").of((t) => t.as_indexed((i) => i.of(Node)))).update();
  const store = R.store([...SCHEMAS, ...KINDS, Node]);
  const catalog = store.singleton("Schemas.Catalog") as unknown as R.Catalog;
  const snapshot = JSON.ToJSON(store).Reachable(R.Catalog.Schema, catalog as never);
  assert(snapshot.split('"name": "Phone"').length - 1 === 1); // shared: written once, by every schema that refers to it
  const back = JSON.FromJSON(R.store()).Reachable(R.Catalog.Schema, snapshot) as unknown as R.Catalog;
  assert(same(back.schemas.map((s: any) => s.name), catalog.schemas.map((s: any) => s.name)));
  assert(catalog.schemas.every((original, i) => original !== back.schemas[i]));
  assert(JSON.ToJSON(R.store()).Reachable(R.Catalog.Schema, back as never) === snapshot);
  const modules = new Proxies.OfStore();
  assert(JSON.ToJSON(modules)(S.Module.Schema, Modules.module(modules, back.schemas as never) as never) === JSON.ToJSON(modules)(
    S.Module.Schema, Modules.module(modules, catalog.schemas as never) as never)); // the same schemas
  const copy = back.schemas[back.schemas.length - 1] as S.OfObject.Data;
  assert((copy.properties.get("next")?.type as S.OfIndexed.Data).item === copy); // recursive: the node's list holds nodes
  assert((back.schemas[0] as S.OfObject.Data).properties.get("home")?.type === back.schemas[1]); // shared: one Phone
  const byte = back.schemas[6] as S.OfApply.Data;
  assert(equal(byte.arguments, new Map([["n", 8n]])) && byte.of === back.schemas[3]); // Byte applies Word
  const word = back.schemas[3] as S.OfNative.Data;
  assert(n.equals(word.bits) && new S.OfNative.Data(BigInt).equals(word.parameters.get("n")?.type));
}

## RFL-04 · Validated, built and updated as any objects; a native another format names cannot be read

In [ ]:
{
  const store: any = R.store([...SCHEMAS, ...KINDS]);
  assert(V.Validate(store).Reachable(R.Catalog.Schema, store.singleton("Schemas.Catalog")).length === 0);
  const home = Contact.properties.get("home") as S.OfProperty.Data;
  const builder = store.builder("Schemas.Property", home);
  builder.property("description", (p: any) => p.value((a: any) => a.as_native((v: any) => v.set("The home phone"))));
  assert(builder.update() === home && home.description === "The home phone" && home.type === Phone); // the schema itself
  const fresh = store.builder("Schemas.Property");
  fresh.property("name", (p: any) => p.value((a: any) => a.as_native((v: any) => v.set("fax"))));
  const made = fresh.create();
  assert(made instanceof S.OfProperty.Data && made.name === "fax" && made.type === null);
  assert(same(store.builder("Schemas.Catalog", store.singleton("Schemas.Catalog")).update().schemas, [...SCHEMAS, ...KINDS]));
  const foreign = S.OfNative.resolve((t) => t.token("ccpp", "int32_t"));
  assert(same(R.store([foreign]).extent("Schemas.Native"), [foreign]));
  raises(TypeError, () => V.properties_of(new S.OfNative.Data("x") as never), "unsupported native type 'x'");
}